In [13]:
import mlflow
mlflow.set_tracking_uri('http://ec2-56-228-23-191.eu-north-1.compute.amazonaws.com:5000/')

In [14]:
mlflow.set_experiment("Exp3 TfIdf Bigram max_features")

2026/03/14 14:00:45 INFO mlflow.tracking.fluent: Experiment with name 'Exp3 TfIdf Bigram max_features' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://mlflow-bucket-1807077/6', creation_time=1773475245573, experiment_id='6', last_update_time=1773475245573, lifecycle_stage='active', name='Exp3 TfIdf Bigram max_features', tags={}>

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

In [16]:
df=pd.read_csv("processed_data.csv")
df['clean_comment'] = df['clean_comment'].fillna('')
df = df[['clean_comment', 'category']]
df.head()

,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,-1
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1


In [ ]:
# Step 1: Function to run the experiment
def run_experiment_tfidf_max_features(max_features):

    ngram_range = (1, 2)  # Bigram setting

    # Step 2: Vectorization using TF-IDF with varying max_features
    vectorizer = TfidfVectorizer(
        ngram_range=ngram_range,
        max_features=max_features
    )

    X_train, X_test, y_train, y_test = train_test_split(
        df['clean_comment'],
        df['category'],
        test_size=0.2,
        random_state=42
    )

    X_train = vectorizer.fit_transform(X_train)
    X_test = vectorizer.transform(X_test)

    # Step 4: Define and train a Random Forest model
    with mlflow.start_run() as run:

        # Set tags for the experiment and run
        mlflow.set_tag("mlflow.runName", f"TFIDF_Bigrams_max_features_{max_features}")
        mlflow.set_tag("experiment_type", "feature_engineering")
        mlflow.set_tag("model_type", "RandomForestClassifier")

        # Add a description
        mlflow.set_tag(
            "description",
            f"RandomForest with TF-IDF Bigrams, max_features={max_features}"
        )

        # Log vectorizer parameters
        mlflow.log_param("vectorizer_type", "TF-IDF")
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("vectorizer_max_features", max_features)

        # Log Random Forest parameters
        n_estimators = 200
        max_depth = 15

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)

        # Initialize and train the model
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=42
        )

        model.fit(X_train, y_train)

        # Step 5: Make predictions and log metrics
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(
            y_test,
            y_pred,
            output_dict=True
        )

        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)

        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix: TF-IDF Bigrams, max_features={max_features}")

        plt.savefig("confusion_matrix.png")
        mlflow.log_artifact("confusion_matrix.png")
        plt.close()

        # Log the model
        mlflow.sklearn.log_model(
            model,
            f"random_forest_model_tfidf_Bigrams_{max_features}"
        )


# Step 6: Test various max_features values
max_features_values = [1000, 2000, 3000, 4000, 5000]

for max_features in max_features_values:
    run_experiment_tfidf_max_features(max_features)

/Users/rakibmahmud/Library/Python/3.9/lib/python/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)
2026/03/14 14:01:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/14 14:01:44 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run TFIDF_Bigrams_max_features_1000 at: http://ec2-56-228-23-191.eu-north-1.compute.amazonaws.com:5000/#/experiments/6/runs/77384a9f11124ab88ae0377545e0c0d2
🧪 View experiment at: http://ec2-56-228-23-191.eu-north-1.compute.amazonaws.com:5000/#/experiments/6


2026/03/14 14:02:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/14 14:02:43 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run TFIDF_Bigrams_max_features_2000 at: http://ec2-56-228-23-191.eu-north-1.compute.amazonaws.com:5000/#/experiments/6/runs/e26a8326aefc4b76a2bbe3bda40399e4
🧪 View experiment at: http://ec2-56-228-23-191.eu-north-1.compute.amazonaws.com:5000/#/experiments/6


2026/03/14 14:03:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/14 14:03:44 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


...

/Users/rakibmahmud/Library/Python/3.9/lib/python/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)
2026/03/14 14:05:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/14 14:05:38 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run TFIDF_Bigrams_max_features_5000 at: http://ec2-56-228-23-191.eu-north-1.compute.amazonaws.com:5000/#/experiments/6/runs/0d15c9f16ee34b82a93c49b5f72c63a0
🧪 View experiment at: http://ec2-56-228-23-191.eu-north-1.compute.amazonaws.com:5000/#/experiments/6


2026/03/14 14:06:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/14 14:06:45 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run TFIDF_Bigrams_max_features_6000 at: http://ec2-56-228-23-191.eu-north-1.compute.amazonaws.com:5000/#/experiments/6/runs/c79d6a2d087842429017c4136b3e2af4
🧪 View experiment at: http://ec2-56-228-23-191.eu-north-1.compute.amazonaws.com:5000/#/experiments/6


MlflowException: API request to http://ec2-56-228-23-191.eu-north-1.compute.amazonaws.com:5000/api/2.0/mlflow/runs/get failed with exception HTTPConnectionPool(host='ec2-56-228-23-191.eu-north-1.compute.amazonaws.com', port=5000): Max retries exceeded with url: /api/2.0/mlflow/runs/get?run_uuid=9b80d6fdd0c940ba809b6ca37455097d&run_id=9b80d6fdd0c940ba809b6ca37455097d (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x13455ed30>: Failed to establish a new connection: [Errno 61] Connection refused'))